# 01.7 · Solutions

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/01-working-with-data/07-plotting/solutions.ipynb)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
sns.set_theme(style="whitegrid")

In [ ]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import sys, pandas as pd
sys.path.insert(0, str(data_path("download.py").parent))
from download import ensure_online_retail

raw = pd.read_excel(ensure_online_retail(), sheet_name="Year 2010-2011")
print(raw.shape)
raw.head()

In [ ]:
def clean_retail(df):
    """The standard ShopWave cleaning used from lesson 01.5 onwards."""
    out = df.copy()
    out = out[~out["Invoice"].astype(str).str.startswith("C")]   # drop cancellations
    out = out[(out["Quantity"] > 0) & (out["Price"] > 0)]         # drop returns / free items
    out = out.dropna(subset=["Customer ID"])                       # drop guest checkouts
    out = out.drop_duplicates()
    out["Customer ID"] = out["Customer ID"].astype(int)
    out["Revenue"] = out["Quantity"] * out["Price"]
    return out

sales = clean_retail(raw)
print(f"{len(raw):,} rows → {len(sales):,} after cleaning")

In [ ]:
sales["month"] = sales["InvoiceDate"].dt.to_period("M").dt.to_timestamp()
monthly = sales.groupby("month")["Revenue"].sum()
by_country = sales.groupby("Country")["Revenue"].sum().sort_values(ascending=False)
order_values = sales.groupby("Invoice")["Revenue"].sum()
customers = sales.groupby("Customer ID").agg(
    orders=("Invoice", "nunique"), spend=("Revenue", "sum"),
    products=("StockCode", "nunique"), items=("Quantity", "sum"),
    first_order=("InvoiceDate", "min"), last_order=("InvoiceDate", "max"))
customers["days_active"] = (customers["last_order"] - customers["first_order"]).dt.days
customers["avg_order"] = customers["spend"] / customers["orders"]

## Exercise 1

In [ ]:
non_uk = by_country.drop("United Kingdom").index[:5]
grid = sales[sales["Country"].isin(non_uk)].pivot_table(index="month", columns="Country", values="Revenue", aggfunc="sum", fill_value=0)
fig, ax = plt.subplots(figsize=(9, 4))
grid.plot(ax=ax, marker="o")
ax.set(title="Non-UK revenue by month", xlabel="", ylabel="£")
plt.show()

All five peak in autumn 2011. The Netherlands and EIRE are lumpy (a few big wholesale orders); Germany and France are steadier. "Growing fastest" is ambiguous when one order can double a month, which is exactly why you plot before you compute growth rates.

## Exercise 2

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.5))
sns.histplot(sales["Quantity"], bins=50, ax=ax1); ax1.set(title="linear")
sns.histplot(sales["Quantity"], bins=50, log_scale=True, ax=ax2); ax2.set(title="log")
plt.tight_layout(); plt.show()
print(sales["Quantity"].quantile([.5, .9, .99, .999]))

On the linear scale one quantity of 80,995 forces the axis to 80,000 and every real order sits in the first bar. The log scale shows the actual structure: peaks at 1, 2, 6, 12, 24 (pack sizes).

## Exercise 3

In [ ]:
cust_country = sales.groupby("Customer ID")["Country"].first()
cs = customers.join(cust_country)
fig, ax = plt.subplots(figsize=(9, 4))
sns.boxplot(data=cs[cs["Country"].isin(by_country.index[:6])], x="Country", y="spend", ax=ax, color="lightsteelblue")
ax.set_yscale("log"); ax.set(title="Customer spend by country", xlabel="", ylabel="£ (log)")
plt.show()
cs[cs["Country"] == "Netherlands"].sort_values("spend", ascending=False).head(3)

The Netherlands has the widest spread: one customer (14646) spent about £280k, far above the rest. Not an error; a wholesaler. Whether to keep such customers in a *retail* model is a business decision you now know to raise.

## Exercise 4

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.scatterplot(data=customers, x="orders", y="products", alpha=.3, s=15, ax=ax)
ax.set(xscale="log", yscale="log", title="Orders vs distinct products per customer")
plt.show()
print("correlation:", customers["orders"].corr(customers["products"]).round(2))

A tight upward band: customers who order more often also buy a wider range. The two columns carry much of the same information (correlation about 0.7); the heatmap in the lesson said the same thing in one cell.